Imports

In [14]:
import matplotlib.pyplot as plt
from onnx.tools.net_drawer import GetPydotGraph, GetOpNodeProducer
import numpy
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, Dataset, train as train_lgbm
import onnxruntime as rt
from onnxconverter_common.data_types import FloatTensorType
from onnxmltools.convert import convert_lightgbm
import IPython.display as d

Load Data

In [15]:
import pandas as pd

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y)
clr = LGBMClassifier(verbosity=-1)
clr.fit(X_train, y_train)

d.display(pd.DataFrame(X, columns=[iris.feature_names]))
d.display(clr)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


LGBMClassifier(verbosity=-1)

Convert model to ONNX

In [16]:
initial_type = [("float_input", FloatTensorType([None, 4]))]
onx = convert_lightgbm(clr, initial_types=initial_type, decision_path=True, decision_leaf=True)
display(onx)

The maximum opset needed by this model is only 9.


ir_version: 4
producer_name: "OnnxMLTools"
producer_version: "1.14.0"
domain: "onnxconverter-common"
model_version: 0
doc_string: ""
graph {
  node {
    input: "float_input"
    output: "dpath1"
    name: "TreeEnsembleClassifier1"
    op_type: "TreeEnsembleRegressor"
    attribute {
      name: "n_targets"
      i: 1
      type: INT
    }
    attribute {
      name: "nodes_falsenodeids"
      ints: 2
      ints: 0
      ints: 4
      ints: 0
      ints: 0
      ints: 2
      ints: 0
      ints: 4
      ints: 0
      ints: 6
      ints: 0
      ints: 0
      ints: 2
      ints: 4
      ints: 6
      ints: 0
      ints: 0
      ints: 0
      ints: 0
      ints: 2
      ints: 0
      ints: 4
      ints: 6
      ints: 0
      ints: 0
      ints: 0
      ints: 2
      ints: 0
      ints: 4
      ints: 0
      ints: 6
      ints: 0
      ints: 0
      ints: 2
      ints: 4
      ints: 6
      ints: 0
      ints: 0
      ints: 0
      ints: 0
      ints: 2
      ints: 0
      ints: 4
      i

Predict with native lgbm

In [17]:
native_pred=clr.predict(X_test, pred_leaf=True)
display(native_pred)

array([[2, 1, 3, ..., 3, 2, 0],
       [1, 1, 3, ..., 1, 2, 0],
       [0, 0, 3, ..., 2, 0, 2],
       ...,
       [2, 1, 3, ..., 3, 2, 3],
       [2, 1, 3, ..., 3, 2, 1],
       [2, 2, 1, ..., 3, 1, 1]], dtype=int32)

Predict in onx session

In [19]:
sess = rt.InferenceSession(onx.SerializeToString(), providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
pred_onx = sess.run(['label', 'decision_path', 'decision_leaf'], {input_name: X_test.astype(numpy.float32)})
d.display(pd.DataFrame(pred_onx[2]))

2025-02-18 11:55:23.291704 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {1} does not match actual shape of {38} for output label


,0
0,1573
1,1550
2,1167
3,1468
4,1251
5,1516
6,1070
7,1070
8,1396
9,1491
